# 05 - Backtest Robusto da Estratégia

Validação completa da estratégia 1-2-4-1-2-4:
1. Walk-forward backtest nos 3.9M registros
2. Simulação Monte Carlo (1000 cenários)
3. Análise de sensibilidade
4. Métricas: Sharpe, drawdown, profit factor

In [ ]:
import sys
sys.path.insert(0, '.')
from config_analysis import *

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import plotly.io as pio
pio.templates.default = 'plotly_dark'

CYAN, MAGENTA, GREEN, RED, YELLOW = '#00f0ff', '#ff00ff', '#00ff88', '#ff3366', '#ffff00'

df = load_processed_data()
if df is None:
    raise FileNotFoundError("Execute o notebook 01 primeiro!")

print(f"Dataset: {len(df):,} registros")

## 1. Motor de Backtest

In [ ]:
def backtest_strategy(
    multiplicadores: np.ndarray,
    trigger: int = 6,
    target: float = 2.0,
    pattern: list = [1, 2, 4, 1, 2, 4],
    break_at: int = 12,
    unit: float = 1.0,
) -> dict:
    """Executa backtest completo da estratégia."""
    n = len(multiplicadores)
    is_low = (multiplicadores < 2.0).astype(int)
    
    # Calcular streaks
    streaks = np.zeros(n, dtype=int)
    for i in range(1, n):
        if is_low[i] == 1:
            streaks[i] = streaks[i-1] + 1
    
    # Simulação
    equity = [0.0]
    trades = []
    max_equity = 0.0
    max_drawdown = 0.0
    wins = 0
    losses = 0
    
    i = 0
    while i < n - 1:
        if streaks[i] >= trigger:
            dobra = streaks[i] - trigger
            if dobra < len(pattern):
                bet = unit * pattern[dobra]
                next_mult = multiplicadores[i + 1]
                
                if next_mult >= target:
                    pnl = bet * (target - 1)
                    wins += 1
                else:
                    pnl = -bet
                    losses += 1
                
                current = equity[-1] + pnl
                equity.append(current)
                trades.append({
                    'index': i,
                    'streak': streaks[i],
                    'dobra': dobra,
                    'bet': bet,
                    'next_mult': next_mult,
                    'hit': next_mult >= target,
                    'pnl': pnl,
                    'equity': current,
                })
                
                # Drawdown
                max_equity = max(max_equity, current)
                dd = max_equity - current
                max_drawdown = max(max_drawdown, dd)
        i += 1
    
    total = wins + losses
    win_rate = wins / total if total > 0 else 0
    
    # Profit factor
    gross_profit = sum(t['pnl'] for t in trades if t['pnl'] > 0)
    gross_loss = abs(sum(t['pnl'] for t in trades if t['pnl'] < 0))
    profit_factor = gross_profit / gross_loss if gross_loss > 0 else float('inf')
    
    # Sharpe (diário aproximado)
    if len(trades) > 1:
        returns = [t['pnl'] for t in trades]
        sharpe = np.mean(returns) / np.std(returns) * np.sqrt(252) if np.std(returns) > 0 else 0
    else:
        sharpe = 0
    
    return {
        'equity': equity,
        'trades': trades,
        'total_trades': total,
        'wins': wins,
        'losses': losses,
        'win_rate': win_rate,
        'final_equity': equity[-1],
        'max_drawdown': max_drawdown,
        'profit_factor': profit_factor,
        'sharpe': sharpe,
        'gross_profit': gross_profit,
        'gross_loss': gross_loss,
    }

print("Motor de backtest pronto.")

## 2. Backtest Principal (3.9M registros)

In [ ]:
mults = df['multiplicador'].values

result = backtest_strategy(
    mults,
    trigger=STRATEGY_TRIGGER,
    target=STRATEGY_TARGET,
    pattern=STRATEGY_PATTERN,
    break_at=STRATEGY_BREAK_AT,
)

print("=" * 60)
print(f"BACKTEST: Estratégia {STRATEGY_PATTERN}")
print(f"Trigger: {STRATEGY_TRIGGER} LOWs | Target: {STRATEGY_TARGET}x")
print("=" * 60)
print(f"  Total trades:    {result['total_trades']:,}")
print(f"  Wins:            {result['wins']:,}")
print(f"  Losses:          {result['losses']:,}")
print(f"  Win Rate:        {result['win_rate']*100:.2f}%")
print(f"  Equity Final:    {result['final_equity']:+,.2f} unidades")
print(f"  Max Drawdown:    {result['max_drawdown']:,.2f} unidades")
print(f"  Profit Factor:   {result['profit_factor']:.3f}")
print(f"  Sharpe Ratio:    {result['sharpe']:.3f}")
print(f"  Lucro Bruto:     {result['gross_profit']:+,.2f}")
print(f"  Perda Bruta:     {result['gross_loss']:,.2f}")

In [ ]:
# Curva de Equity
fig = go.Figure()
fig.add_trace(go.Scatter(
    y=result['equity'],
    mode='lines', line=dict(color=GREEN, width=1.5),
    fill='tozeroy', fillcolor='rgba(0,255,136,0.05)',
    name='Equity'
))
fig.add_hline(y=0, line_dash='dash', line_color=RED, opacity=0.5)

# Marcar max drawdown
eq = np.array(result['equity'])
running_max = np.maximum.accumulate(eq)
drawdowns = running_max - eq
max_dd_idx = np.argmax(drawdowns)
fig.add_annotation(
    x=max_dd_idx, y=eq[max_dd_idx],
    text=f'Max DD: -{result["max_drawdown"]:.1f}',
    showarrow=True, arrowcolor=RED,
    font=dict(color=RED)
)

fig.update_layout(
    title='Curva de Equity - Backtest Completo',
    xaxis_title='Trade #', yaxis_title='Equity (unidades)',
    height=500
)
fig.show()

## 3. Análise de Sensibilidade (Trigger)

In [ ]:
# Variar trigger de 4 a 12
sensibility = []
for trig in range(4, 13):
    r = backtest_strategy(mults, trigger=trig, target=STRATEGY_TARGET, pattern=STRATEGY_PATTERN)
    sensibility.append({
        'trigger': trig,
        'trades': r['total_trades'],
        'win_rate': r['win_rate'] * 100,
        'equity': r['final_equity'],
        'max_dd': r['max_drawdown'],
        'profit_factor': r['profit_factor'],
        'sharpe': r['sharpe'],
    })

sens_df = pd.DataFrame(sensibility)

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Win Rate', 'Equity Final', 'Max Drawdown', 'Profit Factor')
)

fig.add_trace(go.Bar(
    x=sens_df['trigger'], y=sens_df['win_rate'],
    marker_color=CYAN, name='Win Rate'
), row=1, col=1)

fig.add_trace(go.Bar(
    x=sens_df['trigger'], y=sens_df['equity'],
    marker_color=[GREEN if e > 0 else RED for e in sens_df['equity']],
    name='Equity'
), row=1, col=2)

fig.add_trace(go.Bar(
    x=sens_df['trigger'], y=sens_df['max_dd'],
    marker_color=RED, name='Max DD'
), row=2, col=1)

fig.add_trace(go.Bar(
    x=sens_df['trigger'], y=sens_df['profit_factor'],
    marker_color=YELLOW, name='PF'
), row=2, col=2)

fig.update_layout(height=700, title_text='Análise de Sensibilidade: Trigger 4-12')
fig.show()

print("\nTabela de sensibilidade:")
print(sens_df.to_string(index=False))

## 4. Análise de Sensibilidade (Target)

In [ ]:
# Variar target de 1.5 a 3.0
targets = np.arange(1.5, 3.1, 0.1)
target_sens = []

for tgt in targets:
    r = backtest_strategy(mults, trigger=STRATEGY_TRIGGER, target=round(tgt, 1), pattern=STRATEGY_PATTERN)
    target_sens.append({
        'target': round(tgt, 1),
        'trades': r['total_trades'],
        'win_rate': r['win_rate'] * 100,
        'equity': r['final_equity'],
        'max_dd': r['max_drawdown'],
        'profit_factor': r['profit_factor'],
    })

tgt_df = pd.DataFrame(target_sens)

fig = make_subplots(specs=[[{'secondary_y': True}]])
fig.add_trace(go.Scatter(
    x=tgt_df['target'], y=tgt_df['equity'],
    mode='lines+markers', name='Equity',
    line=dict(color=GREEN, width=2),
), secondary_y=False)
fig.add_trace(go.Scatter(
    x=tgt_df['target'], y=tgt_df['win_rate'],
    mode='lines+markers', name='Win Rate %',
    line=dict(color=CYAN, width=2),
), secondary_y=True)

fig.update_layout(title='Sensibilidade ao Target', height=450)
fig.update_yaxes(title_text='Equity', secondary_y=False)
fig.update_yaxes(title_text='Win Rate %', secondary_y=True)
fig.show()

print("\nTabela de sensibilidade (target):")
print(tgt_df.to_string(index=False))

## 5. Simulação Monte Carlo

In [ ]:
# Monte Carlo: embaralhar a série e comparar com o resultado real
n_simulations = 1000
mc_equities = []

print(f"Rodando {n_simulations} simulações Monte Carlo...")
rng = np.random.default_rng(42)

for sim in range(n_simulations):
    # Embaralhar multiplicadores (destrói padrões temporais)
    shuffled = rng.permutation(mults)
    r = backtest_strategy(
        shuffled, trigger=STRATEGY_TRIGGER, target=STRATEGY_TARGET, pattern=STRATEGY_PATTERN
    )
    mc_equities.append(r['final_equity'])
    if (sim + 1) % 200 == 0:
        print(f"  {sim+1}/{n_simulations} concluídas")

mc_equities = np.array(mc_equities)
real_equity = result['final_equity']

# Percentil do resultado real
percentile = (mc_equities < real_equity).mean() * 100

print(f"\n{'='*50}")
print(f"MONTE CARLO ({n_simulations} simulações)")
print(f"{'='*50}")
print(f"  Equity real:      {real_equity:+,.2f}")
print(f"  MC média:         {mc_equities.mean():+,.2f}")
print(f"  MC mediana:       {np.median(mc_equities):+,.2f}")
print(f"  MC std:           {mc_equities.std():,.2f}")
print(f"  MC min:           {mc_equities.min():+,.2f}")
print(f"  MC max:           {mc_equities.max():+,.2f}")
print(f"  Percentil real:   {percentile:.1f}%")
print()
if percentile > 95:
    print(f"  O resultado real está no TOP {100-percentile:.1f}% → MELHOR que aleatório")
elif percentile < 5:
    print(f"  O resultado real está no BOTTOM {percentile:.1f}% → PIOR que aleatório")
else:
    print(f"  O resultado real é compatível com aleatoriedade (percentil {percentile:.1f}%)")

In [ ]:
# Visualizar distribuição Monte Carlo
fig = go.Figure()
fig.add_trace(go.Histogram(
    x=mc_equities, nbinsx=50,
    marker_color=CYAN, opacity=0.7,
    name='Monte Carlo'
))
fig.add_vline(
    x=real_equity, line_dash='dash', line_color=GREEN, line_width=3,
    annotation_text=f'Real: {real_equity:+,.0f} (P{percentile:.0f})'
)
fig.add_vline(
    x=mc_equities.mean(), line_dash='dot', line_color=YELLOW,
    annotation_text=f'MC Média: {mc_equities.mean():+,.0f}'
)

fig.update_layout(
    title=f'Monte Carlo: Distribuição de {n_simulations} Simulações',
    xaxis_title='Equity Final (unidades)',
    yaxis_title='Frequência',
    height=500
)
fig.show()

## 6. Walk-Forward Analysis

In [ ]:
# Dividir em períodos mensais e calcular equity de cada mês
df['ano_mes'] = df['date'].dt.strftime('%Y-%m')
months = sorted(df['ano_mes'].unique())

monthly_results = []
for month in months:
    mask = df['ano_mes'] == month
    month_mults = df.loc[mask, 'multiplicador'].values
    if len(month_mults) < 100:
        continue
    r = backtest_strategy(
        month_mults, trigger=STRATEGY_TRIGGER, target=STRATEGY_TARGET, pattern=STRATEGY_PATTERN
    )
    monthly_results.append({
        'month': month,
        'trades': r['total_trades'],
        'win_rate': r['win_rate'] * 100,
        'equity': r['final_equity'],
        'max_dd': r['max_drawdown'],
    })

mres_df = pd.DataFrame(monthly_results)

fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=('Equity por Mês', 'Win Rate por Mês'))

fig.add_trace(go.Bar(
    x=mres_df['month'], y=mres_df['equity'],
    marker_color=[GREEN if e > 0 else RED for e in mres_df['equity']],
    name='Equity'
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=mres_df['month'], y=mres_df['win_rate'],
    mode='lines+markers', name='Win Rate',
    line=dict(color=CYAN, width=2)
), row=2, col=1)
fig.add_hline(y=50, line_dash='dash', line_color=YELLOW, row=2, col=1)

fig.update_layout(height=700, title_text='Walk-Forward: Performance Mensal')
fig.show()

# Estatísticas mensais
pos_months = (mres_df['equity'] > 0).sum()
total_months = len(mres_df)
print(f"\nMeses positivos: {pos_months}/{total_months} ({pos_months/total_months*100:.1f}%)")
print(f"Meses negativos: {total_months - pos_months}/{total_months}")
print(f"Melhor mês: {mres_df.loc[mres_df['equity'].idxmax(), 'month']} ({mres_df['equity'].max():+,.1f})")
print(f"Pior mês:   {mres_df.loc[mres_df['equity'].idxmin(), 'month']} ({mres_df['equity'].min():+,.1f})")

## 7. Resumo Final do Backtest

In [ ]:
print("=" * 60)
print("RESUMO FINAL DO BACKTEST")
print("=" * 60)
print(f"""
ESTRATÉGIA: {STRATEGY_PATTERN}
TRIGGER: {STRATEGY_TRIGGER} LOWs consecutivos
TARGET: {STRATEGY_TARGET}x
DADOS: {len(df):,} registros ({df['date'].min().strftime('%Y-%m')} a {df['date'].max().strftime('%Y-%m')})

RESULTADO PRINCIPAL:
  Total Trades:     {result['total_trades']:,}
  Win Rate:         {result['win_rate']*100:.2f}%
  Equity Final:     {result['final_equity']:+,.2f} unidades
  Max Drawdown:     {result['max_drawdown']:,.2f} unidades
  Profit Factor:    {result['profit_factor']:.3f}
  Sharpe Ratio:     {result['sharpe']:.3f}

WALK-FORWARD:
  Meses positivos:  {pos_months}/{total_months} ({pos_months/total_months*100:.1f}%)
  Equity mensal média: {mres_df['equity'].mean():+,.2f}

MONTE CARLO ({n_simulations} sims):
  Percentil real:   {percentile:.1f}%
  O resultado é {'SIGNIFICATIVAMENTE MELHOR' if percentile > 95 else 'SIGNIFICATIVAMENTE PIOR' if percentile < 5 else 'COMPATÍVEL COM'} vs aleatório

SENSIBILIDADE:
  Melhor trigger:   {sens_df.loc[sens_df['equity'].idxmax(), 'trigger']} (equity: {sens_df['equity'].max():+,.1f})
  Melhor target:    {tgt_df.loc[tgt_df['equity'].idxmax(), 'target']}x (equity: {tgt_df['equity'].max():+,.1f})
""")